# Redis

Dominio: cadena de fábricas de pastas con franquicias (mismo dataset que el resto del TP, en `../data/`).

Usamos Redis para lo que conviene tener en memoria: lecturas por ID muy frecuentes, colas y datos que vencen solos.

## Conexión y carga de datos

Redis corre en Docker (puerto 6379). Con decode_responses=True nos devuelve strings en vez de bytes.

In [ ]:
import json
import time
import pandas as pd
import redis

# conexión a Redis (mismo host/puerto que en docker-compose.yml).
r = redis.Redis(host="localhost", port=6379, db=0, decode_responses=True)
r.ping()  # lanza error si Redis no está levantado
r.flushdb()  #hacemos flushdb() al inicio para arrancar siempre con la base vacia y que la notebook sea reproducible.
print("Conectado a Redis y base limpia.")

In [ ]:
# cargamos los datos

clientes = pd.read_csv("../data/cliente.csv")
pastas = pd.read_csv("../data/pasta.csv")
detalles = pd.read_csv("../data/detalle_compra.csv")

print(f"clientes: {len(clientes)} | pastas: {len(pastas)} | detalles: {len(detalles)}")
clientes.head(3)

## Modelo clave-valor y hashes

Guardamos tres datos que se consultan seguido de a uno por ID:

- Perfil de cliente, como hash `cliente:{id}`
- Ficha de pasta, como hash `pasta:{sello}:{codigo}`
- Precio por kilo, como string `precio:{sello}:{codigo}`

Usamos hashes para los objetos con varios campos y un string simple para el precio, que es un solo valor que se lee mucho y cambia seguido.

Lo hacemos en Redis y no en PostgreSQL porque estas fichas se piden en cada pantalla y en cada venta. Redis las devuelve enteras con un HGETALL en memoria y funciona como caché de lectura.

In [ ]:

for _, c in clientes.head(5).iterrows():
    r.hset(f"cliente:{c.id_cliente}", mapping={
        "nombre": c.nombre,
        "apellido": c.apellido,
        "email": c.email,
        "telefono": c.telefono,
        "sello": c.sello,
    })


for _, p in pastas.head(5).iterrows():
    clave_pasta = f"pasta:{p.sello}:{p.codigo_pasta}"
    r.hset(clave_pasta, mapping={
        "nombre": p.nombre,
        "tipo": p.tipo,
        "precio_por_kilo": p.precio_por_kilo,
    })
    r.set(f"precio:{p.sello}:{p.codigo_pasta}", p.precio_por_kilo)

print("Datos cargados en Redis (clientes, pastas y precios).")

In [ ]:

print("Cliente 1:", r.hgetall("cliente:1"))

print("Nombre pasta FR-001:P001:", r.hget("pasta:FR-001:P001", "nombre"))

print("Precio FR-001:P001:", r.get("precio:FR-001:P001"))

In [ ]:

clave = "pasta:FR-001:P001"
print("Precio ANTES:", r.hget(clave, "precio_por_kilo"))

# actualizamos el precio en el hash y también en el string simple.
r.hset(clave, "precio_por_kilo", 3800.00)
r.set("precio:FR-001:P001", 3800.00)

print("Precio DESPUÉS:", r.hget(clave, "precio_por_kilo"))

## Listas como cola

Armamos una cola de producción FIFO por franquicia.

- Los pedidos nuevos entran por el final con `RPUSH`.
- La cocina toma el siguiente desde el inicio con `LPOP`.

In [ ]:

cola = "cola_produccion:FR-001"
r.delete(cola)  # por si se re-ejecuta la celda

# tomamos algunos renglones de detalle_compra de la franquicia FR-001 como pedidos.
pedidos = detalles[detalles.sello == "FR-001"].head(5)
for _, d in pedidos.iterrows():
    pedido = json.dumps({"id_compra": int(d.id_compra),
                          "pasta": d.codigo_pasta,
                          "kg": float(d.cantidad_kg)})
    r.rpush(cola, pedido)

print(f"Pedidos en cola: {r.llen(cola)}")
for p in r.lrange(cola, 0, -1):
    print("  ", p)

In [ ]:
# simulacion

nuevo = json.dumps({"id_compra": 9999, "pasta": "P002", "kg": 2.5})
r.rpush(cola, nuevo)
print(f"Entró un pedido nuevo. La cola tiene {r.llen(cola)} pendientes.")

for _ in range(3):
    siguiente = r.lpop(cola)
    print(f"Cocina produciendo: {siguiente}  ->  quedan {r.llen(cola)} pendientes")

## TTL: datos con tiempo de vida

Tres datos que vencen solos, con tiempos distintos segun para qu sirven:

Guardamos tres datos que se borran solos despues de un tiempo:

- Reserva de un pedido `(reserva:{sello}:{id_compra})`: 10 minutos. Si el cliente no confirma, se libera sola.
- Promoción `(promo:{sello}:{codigo_pasta})`: 1 hora. Es una oferta que se vence.
- Caché de un ranking `(cache:top_pastas:{sello})`: 5 minutos.

Lo bueno del TTL es que el dato expira solo, sin tener que borrarlo a mano.

In [ ]:
r.set("reserva:FR-001:1001", "cliente:1 | P001 x 3.5 kg", ex=600)   # 10 min: pedido en armado
r.set("promo:FR-001:P001", "20% OFF", ex=3600)                       # 1 h: oferta acotada
r.set("cache:top_pastas:FR-001", "[P001, P003, P024]", ex=300)       # 5 min: ranking cacheado

# verificamos el tiempo restante
for clave in ["reserva:FR-001:1001", "promo:FR-001:P001", "cache:top_pastas:FR-001"]:
    print(f"{clave}: vigente por {r.ttl(clave)} segundos")

In [ ]:
#  simulacion

restante = r.ttl("reserva:FR-001:1001")
print(f"La reserva del pedido 1001 sigue vigente por {restante} segundos.")

r.set("reserva:FR-001:9999", "cliente:7 | P003 x 2 kg", ex=2)  
print("Reserva 9999 creada, TTL:", r.ttl("reserva:FR-001:9999"), "s. Esperando a que expire...")
time.sleep(3)

if r.ttl("reserva:FR-001:9999") == -2:
    print("La reserva 9999 expiró: los kilos quedan liberados para otros pedidos.")
else:
    print("La reserva 9999 sigue vigente:", r.get("reserva:FR-001:9999"))